# Arm Movement Analysis

Author: Suraj Puvvadi

Last Updated: *06-08-26*

This notebook provides a complete analysis pipeline for arm-swing biomechanics using XSENS motion capture data exported. It loads three-dimensional segment positions and accelerations of the right and left forearms and feet, and pelvis, applies low-pass filtering, and engineers a comprehensive set of kinematic upper limb features describing arm motion and symmetry during gait.

<h2>Imports</h2>

In [44]:
import os
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, butter, filtfilt
from pathlib import Path

<h2>Pre-Processing, Filtering, and Pipeline Configuration</h2>

Several parameters established below govern accelerometer signal processing throughout the pipeline and therefore influence all downstream calculations. The raw XSENS accelerometer and position data used as input were sampled at 60 Hz. Preprocessing the data appropriately is necessary to ensure that only the relevant portions of the raw data are utilized when engineering upper limb movement features.

Previous biomechanical studies have commonly applied low-pass Butterworth filters with cutoff frequencies between 3-10 Hz for upper-limb movement analysis, including 3-6 Hz (https://www.umass.edu/locomotion/pdfs/job-2008.pdf), 6 Hz with a fourth-order Butterworth filter (https://www.mdpi.com/2076-3417/15/4/2177; https://www.sciencedirect.com/science/article/pii/S0021929025005937), 10 Hz with a forward–reverse fourth-order Butterworth filter (https://pmc.ncbi.nlm.nih.gov/articles/PMC8271607/), 3.5 Hz (https://pmc.ncbi.nlm.nih.gov/articles/PMC4549059/), and 3 Hz with a second-order Butterworth filter (https://pmc.ncbi.nlm.nih.gov/articles/PMC7590046/). The Butterworth filter is widely used because it effectively attenuates high-frequency noise while preserving signal characteristics, which essentially ensures that artifacts of the movements aren't distorted due to smoothening while making sure that phase shifts don't persist (https://www.sciencedirect.com/topics/engineering/butterworth-filter; https://www.researchgate.net/profile/Sema-Athamnah/post/Is_there_any_manual_available_teaching_Matlab_for_Applied_Biomechanics_tests/attachment/60dde7926160740001e71d2c/AS%3A1040791648608256%401625155474183/download/BIOMECHANICS_AND_MOTOR_CONTROL_OF_HUMAN.pdf). Specifically, to eliminate phase lag, a two-pass (forward and reverse) implementation of the Butterworth filter is used.

Leveraging established standards in prior literature, this pipeline applies a fourth-order, two-pass low-pass Butterworth filter with a 6 Hz cutoff frequency. Accordingly, `default_frame_rate` is established at 60 Hz, the `filter_cutoff_frame_rate` is 6 Hz, and `butterworth_filter_order` for signal smoothening is at the fourth order. Individual arm swings are identified using peak detection methods. Literature suggests that arm-swing frequencies during regular gait are approximately 1–2 Hz (https://pmc.ncbi.nlm.nih.gov/articles/PMC7590046/). Given that an average swing is 1 Hz, a full forwards or a full backwards motion is 0.5 Hz, meaning 0.5 seconds. Since our sensor data is sampled at 60 frames per second, we can reasonably assume that each portion of a swing last about 30 frames. As such, to ensure all peaks are captured and to prevent double-counting, the minimum peak distance is set to 0.5 seconds (30 frames at 60 Hz), corresponding to the approximate duration of a single forward or backward arm excursion within a gait cycle Finally, metrics intended to represent peak performance are calculated using the average of the three highest values within each swing rather than a single maximum value, reducing the influence of anomalous outliers while preserving true peak behavior within arm movement.

In [45]:
# sampling frequency (Hz)
default_frame_rate = 60.0

# low-path cutoff
filter_cutoff_frame_rate = 6.0

# butterworth filter order
butterworth_filter_order = 4

# minimum distance in frames between peaks
peak_dist_bw_frames = 30

# top-k values for "peak" values of arm metrics
top_k = 3


In [46]:
# Butterworth filtering methods

# establish Butterworth filter coefficients based on frequency cutoff and global sensor framerate
def butter_lowpass_coeffs(cutoff_hz, fs, order=4):
    if cutoff_hz is None:
        raise ValueError("No cutoff frequency has been established.")
    
    # Nyquist frequency is a concept that dictates the maximum frequency that can be reasonably used to capture meaningful information about the data
    # Nyquist is always established at 50% of the frame rate; therefore, we never want the cutoff to be above Nyquist frequency. In our case, that's 30 Hz.
    nyq = 0.5 * fs
    if cutoff_hz <= 0 or cutoff_hz >= nyq:
        raise ValueError(f"cutoff_hz must be between 0 and Nyquist ({nyq} Hz).")
    
    # cutoff needs to be normalized to a value between 0 and 1 for the butter function from scipy to work
    normal_cutoff = cutoff_hz / nyq
    
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return b, a


# create Butterworth coefficients and apply it along an axis
def apply_lowpass_filter(data, cutoff_hz=None, fs=None, order=None):
    if cutoff_hz is None:
        cutoff_hz = filter_cutoff_frame_rate
    if fs is None:
        fs = default_frame_rate
    if order is None:
        order = butterworth_filter_order

    # apply Butterworth coefficients and return array
    b, a = butter_lowpass_coeffs(cutoff_hz, fs, order=order)
    arr = np.asarray(data)

    # filtfilt from scipy performs zero-phase filtering in both the forward and reverse directions
    return filtfilt(b, a, arr, axis=1)


<h2>Helper Functions</h2>

Below are a variety of small, lightweight "helper" functions which are used throughout the pipeline for various tasks. The majority of these functions are to ensure that empty arrays/lists/data are handled seamlessly without issue.

In [47]:
# create file paths for all files of interest
def get_files_from_folder(folder_path, ext=".txt"):
    return [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(ext)]

# extract patient ID, trial number, and test type (normal gait vs backwards gait) from file name
def parse_filename(filename):
    # first, normalize accents in names, since rückwärts (backwards gait) has an accent in the name
    fn = unicodedata.normalize('NFC', filename)
    filename_lower = fn.lower()

    # healthy patients IDs all start with "21" and have IDs longer than patients with PD or ataxia
    # setting up this simple heuristic allows us to filter out subject IDs easily
    subject = fn[:6] if fn[:2] == "21" else fn[:3]
    trial_num = 2 if len(fn) > 4 and fn[4] == '2' else 1
    
    if "normaler" in filename_lower:
        test_type = "NG"
    elif "rückwärts" in filename_lower or "rueckw" in filename_lower:
        test_type = "RG"
    else:
        test_type = "Unknown" # based on our file naming architecture, this should never get called
    return subject, trial_num, test_type


# simple safeguard averaging function to calculate mean or standard deviation, or return N/A otherwise, or return count if exists
def mean_or_nan(x):
    return np.nan if (x is None or len(x) == 0) else np.mean(x)


# this function is primary for calculating variability
# since variability is simply a measure of how different each metric per swing is from one another, data from both sides in concatenated into one array to make comparison easier
def safe_concatenate(arr1, arr2):
    # np.atleast_1d ensures that inputted arrays have at least one dimension and are non-empty
    arr1 = np.atleast_1d(arr1)
    arr2 = np.atleast_1d(arr2)
    
    if arr1.size == 0 and arr2.size == 0:
        return np.array([])
    elif arr1.size == 0:
        return arr2
    elif arr2.size == 0:
        return arr1
    else:
        return np.concatenate([arr1, arr2])

# calculate the top-k average to get peak metrics
def top_k_mean(x, k=3):
    if x is None or len(x) == 0:
        return np.nan

    arr = np.sort(np.asarray(x))
    
    # this if block should theoretically never get called on since all metrics should have an arbitrary N number of swings that is substantially larger than the top-K value
    if len(arr) <= k:
        return np.mean(arr)
    
    return np.mean(arr[-k:])

# asymmetry is returned as a tuple, so a simple function is built to separate nondirectional and directional values from each other
def unpack_asym(asym):
    if asym is None:
        return (np.nan, np.nan)
    if isinstance(asym, dict):
        return (asym.get("nondirectional", np.nan), asym.get("directional", np.nan))
    return (asym, asym)

<h2>Arm-Swing Metrics</h2>

These functions compute cycle-by-cycle and summary metrics from arm and pelvis motion data, after peak detection and segmentation. Metrics were selected based on the most commonly reported arm-related metrics in the literature. Asymmetry and variability are calculated for all features.

In [48]:
# before calculating any metrics, we want to first detect distinct arm swings themselves
# since the primary movement in an arm swing is sagittal (forwards and backwards), we will isolate peak detection to the x-axis of the movement
# "pos_axis" that we pass in is the x-axis data
def detect_arm_cycles(pos_axis, height=None, distance=peak_dist_bw_frames, plot=False, side="RFA"):
    peaks, _ = find_peaks(pos_axis, height=height, distance=distance)
    if plot:
        plt.figure()
        plt.plot(pos_axis, label=side)
        plt.plot(peaks, pos_axis[peaks], "x")
        plt.title(f"{side} Peak Detection")
        plt.show()
    return peaks


# we need to know the duration of each cycle to compute all metrics
# this function calculates the duration of each peak by taking the difference of the frames between peaks and converting it to seconds
# mean is the mean cycle duration per person per trial
def calc_cycle_times(peaks, time):
    intervals = np.diff(time[peaks]) if len(peaks)>1 else np.array([])
    return intervals, np.mean(intervals) if len(intervals)>0 else np.nan


# excursion is defined as the total distance that the arm move in each direction during each cycle, using the pelvic position as the reference point
# this function returns an array of excursion values, one per arm swing; downstream, values can be averaged to obtain mean and peak excursion per person
# excursion - lateral, sagittal, vertical
def calc_excursion(pos_axis, peaks):
    excursions = []
    for i in range(len(peaks)-1):
        segment = pos_axis[peaks[i]:peaks[i+1]+1]
        excursions.append(np.max(segment) - np.min(segment))
    return np.array(excursions)

# velocity is estimated by taking the derivative of positional trajectory at consecutive peaks
# for each arm swing, the instantenous velocity is computed using np.gradient
# mean velocities per cycle are calculated and are appended to a list, which is returned
def calc_velocity(pos_axis, dt, peaks):
    velocities = []
    for i in range(len(peaks)-1):
        segment = pos_axis[peaks[i]:peaks[i+1]+1]
        vel_gradient = np.gradient(segment, dt)
        v = np.mean(np.abs(vel_gradient))
        velocities.append(v)
    return np.array(velocities)



# magnitude of maximum acceleration is calculated by computing the magnitude using acceleration across all three axes
# acc magnitude = sqrt(x_acc^2 + y_acc^2 + z_acc^2)
# a magnitude from each cycle is stored, and the maximum value is returned
def calc_peak_acceleration(acc_array, peaks):
    acc_array = np.asarray(acc_array)

    n_cycles = len(peaks) - 1
    peak_accs = []

    # calculate peak acc per cycle (which is defined as the data from one peak to the consecutively following peak)
    for i in range(n_cycles):
        start = peaks[i]
        end = peaks[i+1] + 1
        segment = acc_array[:, start:end]

        # calculate magnitude across all axes
        magnitude = np.linalg.norm(segment, axis=0)
        peak_accs.append(np.max(magnitude))

    return np.array(peak_accs)


# root mean square acceleration is used to essentially understand the overall intensity/vigor of the movement
# instead of taking a simple average, RMS squares the accelerations together and takes the square root, then the mean, to basically weigh the magnitude of negative and positive accelerations equally instead of them cancelling each other out
def calc_rms_acceleration(acc_array, peaks):
    acc_array = np.asarray(acc_array)

    n_cycles = len(peaks) - 1
    rms_accs = []

    for i in range(n_cycles):
        start = peaks[i]
        end = peaks[i+1] + 1  # include endpoint
        segment = acc_array[:, start:end]

        # RMS formula: sqrt(mean(ax^2 + ay^2 + az^2))
        magnitude_squared = np.sum(segment**2, axis=0)
        rms = np.sqrt(np.mean(magnitude_squared))
        rms_accs.append(rms)

    return np.array(rms_accs)


# DOUBLE-CHECK JERK METRIC!!!
# advanced metric (SPARC) - https://pmc.ncbi.nlm.nih.gov/articles/PMC4674971/
# simple metric - derivative of acceleration - https://pmc.ncbi.nlm.nih.gov/articles/PMC8573129/
# hallmark study hogan and sternad 2009 - https://pmc.ncbi.nlm.nih.gov/articles/PMC3470860/
# jerk is a metric that helps quantify movement smoothness, with negative values indicating smoother movement, and positive values indicating "jerky" movement
# jerk value is the rate of change of acceleration, so we will be taking the first-order derivative of the acceleration vectors across consecutive peaks to compute jerk per cycle
# function returns an array of jerk values across all cycles per participant
# Log dimensionless jerk (LDJ), proposed by Hogan and Sternad, is applied below
# formula: LDJ = log((T^5 / D^2) * integral((jerk_x^2 + jerk_y^2 + jerk_z^2) dt)) where T is the cycle duration in seconds and D is the distance between peaks in the sagittal axis (x-axis)
# More negative = smoother, more coordinated movement.
# More positive  = jerkier, more erratic movement.
def calc_log_dimensionless_jerk(acc_array, pos_axis, peaks, dt):
    acc_array = np.asarray(acc_array)
    n_cycles = len(peaks) - 1
    if n_cycles < 1:
        return np.array([])

    ldj_values = []

    for i in range(n_cycles):
        start = peaks[i]
        end = peaks[i + 1] + 1

        seg_acc = acc_array[:, start:end]
        seg_pos = pos_axis[start:end]
        M = seg_acc.shape[1]

        # M is the number of frames. We need at least 5 frames to reliably calculate jerk
        # in our data we have more than 5 frames in every dataset, but this is a fallback in case there is some sort of issue
        if M < 5:
            ldj_values.append(np.nan)
            continue

        # T: cycle duration
        # converting distance between peaks to seconds
        # dt is 1/framerate, so by multiplying the frame difference between subsequent peaks, we get the number of seconds
        T = (peaks[i + 1] - peaks[i]) * dt

        # D: sagittal peak-to-peak excursion per cycle
        # this is the same logic we applied when calculating sag_exc, except its isolated to just the cycle
        D = np.max(seg_pos) - np.min(seg_pos)
        if D <= 0 or np.isclose(D, 0):
            ldj_values.append(np.nan)
            continue

        # jerk: time derivative of each acceleration axis
        jerk_x = np.gradient(seg_acc[0, :], dt)
        jerk_y = np.gradient(seg_acc[1, :], dt)
        jerk_z = np.gradient(seg_acc[2, :], dt)

        # squared magnitude of the 3D jerk vector at each time point
        jerk_sq = jerk_x**2 + jerk_y**2 + jerk_z**2

        # integral of squared jerk over the cycle (trapezoidal rule)
        jerk_integral = np.trapezoid(jerk_sq, dx=dt)

        # now that we have D and T, we can apply the normalization proposed by Hogan & Sternad to make jerk dimensionless
        dimensionless_jerk = (T**5 / D**2) * jerk_integral

        # log normalization helps make the data between healthy, ataxic, and PD patients comparable
        if dimensionless_jerk <= 0:
            ldj_values.append(np.nan)
        else:
            ldj_values.append(np.log(dimensionless_jerk))

    return np.array(ldj_values)



'''
Asymmetry Calculation - http://doi.org/10.1519/SSC.0000000000000371, http://doi.org/10.1038/s41598-018-31151-9
- BAI-1 and SI measures for symmetry index calculation were recommended for evaluating asymmetry of bilateral movements
- BAI-1 was used for this analysis. BAI-1 --> Bilateral Asymmetry Index, Version 1
- BAI-1: asymmetry = 100 * abs((right side - left side))/(right side + left side)
- Directional and non-directional asymmetries were also computed.
- The directional asymmetry indicates which side exhibits greater movement: values > 0 denote greater movement on the right side, whereas values < 0 indicate greater movement on the left side.
- The non-directional asymmetry expresses the overall magnitude of difference between sides, independent of direction.
'''
def asymmetry(arr1, arr2):
    a1 = np.asarray(arr1)
    a2 = np.asarray(arr2)
    rfa_mean = np.mean(a1)
    lfa_mean = np.mean(a2)

    # np.isclose is preferred to ==0 since means may generate non-exact zeroes, but rather very close to zero values (ex - 1e-16)
    if np.isnan(rfa_mean) or np.isnan(lfa_mean) or np.isclose(rfa_mean + lfa_mean, 0):
        return {"nondirectional": np.nan, "directional": np.nan}
    diff = rfa_mean - lfa_mean
    mean_sum = rfa_mean + lfa_mean
    nondirectional_asym = (np.abs(diff)/np.abs(mean_sum)) * 100.0
    directional_asym = (diff/np.abs(mean_sum)) * 100.0
    return {"nondirectional": nondirectional_asym, "directional": directional_asym}


# coefficient of variation
# absolute value is taken for the mean as well, mainly because the jerk value can be negative, so for everything to be interpretable/comparable, the absolute value of the mean must be taken
def variability(arr):
    arr = np.asarray(arr)
    if arr.size == 0:
        return np.nan

    mu = np.mean(arr)
    if np.isclose(mu, 0):
        return np.nan

    return np.std(arr, ddof=1) / np.abs(mu)

## Loading and Correcting Arm Data

These functions handle importing XSENS-exported motion files and aligning arm trajectories relative to the pelvis reference segment.

In [49]:
def load_arm_data(file_path):
    with open(file_path, 'r', encoding='latin-1') as f:
        lines = f.readlines()
    
    # skip lines that have been commented out to form the dataframe with all the sensor data we have in XSENS files
    # then pull into a numpy array
    data_lines = [line for line in lines if not line.lstrip().startswith('#') and line.strip()!='']
    data = [list(map(float, line.strip().split())) for line in data_lines]
    data = np.array(data)
    
    # find comment in header which has framerate to extract frames per second
    # we know everything is sampled at 60 fps, so in case a sensor file header is anomalous, we will default to the 60 fps
    framerate_line = [line for line in lines if line.lower().startswith("# framerate")]
    framerate = default_frame_rate
    if framerate_line:
        try:
            framerate = float(framerate_line[0].split(":")[1].replace("fps","").strip())
        except:
            framerate = default_frame_rate
    
    # the second column in our data is the frame number. By dividing frame number by frame rate, we get the time stamp
    # since 60 Hz sampling rate means 1 timestamp per second, dividing frame number by frame rate gives us the exact time that has passed since task start
    time = data[:,1] / framerate
    
    idx_pos = {"Pelvis": slice(2,5),"LFA": slice(5,8),"RFA": slice(8,11),"LF": slice(11,14),"RF": slice(14,17)}
    idx_acc = {"Pelvis": slice(17,20),"LFA": slice(20,23),"RFA": slice(23,26),"LF": slice(26,29),"RF": slice(29,32)}

    # transpose data for both position and acceleration, so butterworth filtering can be done easily and arm trajectory can be corrected easily as well
    pos_data = {seg: data[:, idx_pos[seg]].T for seg in idx_pos}
    acc_data = {seg: data[:, idx_acc[seg]].T for seg in idx_acc}
    
    return {"time": time, "pos": pos_data, "acc": acc_data, "framerate": framerate}


# correct_arm_trajectory helps understand the right forearm and left forearm movement with respect to the pelvic position
# literature suggests that pelvis position and spinopelvic orientation affects kinematic metrics
# - https://pubmed.ncbi.nlm.nih.gov/28821443/
# therefore, we want to correct trajectory, with respect to pelvis position, per frame
# therefore, for all metrics, using the RFA and LFA positions following this corrections is sufficient to calculate excursion, etc
def correct_arm_trajectory(pos_data, ref_seg="Pelvis", per_frame=True):
    RFA = pos_data["RFA"]
    LFA = pos_data["LFA"]
    ref = pos_data[ref_seg]
    
    if per_frame:
        RFA_rel = RFA - ref
        LFA_rel = LFA - ref
        ref_centered = ref - ref
    else:
        ref_start = ref[:, 0:1] 
        RFA_rel = RFA - ref_start
        LFA_rel = LFA - ref_start
        ref_centered = ref - ref_start
    
    return RFA_rel, LFA_rel, ref_centered

## Full Arm Movement Analysis Pipeline

These functions automate single-file and batch analysis of XSENS motion data, producing quantitative arm-swing metrics for right and left forearms.

In [50]:
def arm_analysis(file_path, plot=False, per_frame_rel=False, cutoff=None):
    data = load_arm_data(file_path)
    
    if data is None:
        raise ValueError(f"load_arm_data returned None for {file_path}")

    dt = 1.0 / data["framerate"]
    pos = data["pos"]
    acc = data["acc"]

    # relativity - establish the right and left forearm positions, in spatial relation to the pelvis so its normalize to some extent
    RFA_pos, LFA_pos, _ = correct_arm_trajectory(pos, per_frame=per_frame_rel)

    effective_cutoff = cutoff if cutoff is not None else filter_cutoff_frame_rate
    RFA_pos_f = apply_lowpass_filter(RFA_pos, cutoff_hz=effective_cutoff, fs=data["framerate"])
    LFA_pos_f = apply_lowpass_filter(LFA_pos, cutoff_hz=effective_cutoff, fs=data["framerate"])
    acc_RFA_f = apply_lowpass_filter(acc["RFA"], cutoff_hz=effective_cutoff, fs=data["framerate"])
    acc_LFA_f = apply_lowpass_filter(acc["LFA"], cutoff_hz=effective_cutoff, fs=data["framerate"])

    # sagittal axis (x) - we want to detect peaks based on front-back 
    RFA_axis = RFA_pos_f[0, :]
    LFA_axis = LFA_pos_f[0, :]

    # peak detection
    peaks_RFA = detect_arm_cycles(RFA_axis, plot=plot) # change to True to plot movement trajectory
    peaks_LFA = detect_arm_cycles(LFA_axis, plot=plot) # change to True to plot movement trajectory

    # cycles & times
    RFA_cycles, avg_RFA_cycle = calc_cycle_times(peaks_RFA, data["time"])
    LFA_cycles, avg_LFA_cycle = calc_cycle_times(peaks_LFA, data["time"])

    # cycle/asymmetry/variability
    cycle_asym = asymmetry(RFA_cycles, LFA_cycles)
    cycle_variability = variability(safe_concatenate(RFA_cycles, LFA_cycles))

    # swing frequency (Hz)
    # Compute per-cycle frequencies first
    RFA_freqs = 1.0 / RFA_cycles if len(RFA_cycles) > 0 else np.array([])
    LFA_freqs = 1.0 / LFA_cycles if len(LFA_cycles) > 0 else np.array([])
    # Mean swing frequency
    swing_freq_RFA = np.mean(RFA_freqs) if len(RFA_freqs) > 0 else np.nan
    swing_freq_LFA = np.mean(LFA_freqs) if len(LFA_freqs) > 0 else np.nan
    # asymmetry based on cycle-wise frequencies
    swing_freq_asym = asymmetry(RFA_freqs, LFA_freqs)
    # Relative cycle-to-cycle variability (coefficient of variation)
    swing_freq_var = variability(safe_concatenate(RFA_freqs, LFA_freqs))

    # peak acceleration stats
    peak_acc_RFA = calc_peak_acceleration(acc_RFA_f, peaks_RFA)
    peak_acc_LFA = calc_peak_acceleration(acc_LFA_f, peaks_LFA)
    peak_acc_asym = asymmetry(peak_acc_RFA, peak_acc_LFA)
    peak_acc_var = variability(safe_concatenate(peak_acc_RFA, peak_acc_LFA))

    # RMS acceleration stats
    rms_acc_RFA = calc_rms_acceleration(acc_RFA_f, peaks_RFA)
    rms_acc_LFA = calc_rms_acceleration(acc_LFA_f, peaks_LFA)
    rms_acc_asym = asymmetry(rms_acc_RFA, rms_acc_LFA)
    rms_acc_var = variability(safe_concatenate(rms_acc_RFA, rms_acc_LFA))

    # sagittal excursion - distance moved from pelvis front to back (aka anterior posterior)
    # [0, :] is used since it represents the x-axis
    sag_exc_RFA = calc_excursion(RFA_pos_f[0, :], peaks_RFA)
    sag_exc_LFA = calc_excursion(LFA_pos_f[0, :], peaks_LFA)
    sag_exc_asym = asymmetry(sag_exc_RFA, sag_exc_LFA)
    sag_exc_var = variability(safe_concatenate(sag_exc_RFA, sag_exc_LFA))

    # lateral excursion - distance moved from pelvis left to right
    # [1, :] is used because it represents y-axis, which is left-right
    lat_exc_RFA = calc_excursion(RFA_pos_f[1, :], peaks_RFA)
    lat_exc_LFA = calc_excursion(LFA_pos_f[1, :], peaks_LFA)
    lat_exc_asym = asymmetry(lat_exc_RFA, lat_exc_LFA)
    lat_exc_var = variability(safe_concatenate(lat_exc_RFA, lat_exc_LFA))

    # vertical excursion - distance moved from pelvis up to down
    # [2, :] is used since it represents the z-axis
    vert_exc_RFA = calc_excursion(RFA_pos_f[2, :], peaks_RFA)
    vert_exc_LFA = calc_excursion(LFA_pos_f[2, :], peaks_LFA)
    vert_exc_asym = asymmetry(vert_exc_RFA, vert_exc_LFA)
    vert_exc_var = variability(safe_concatenate(vert_exc_RFA, vert_exc_LFA))

    # velocity
    vel_RFA = calc_velocity(RFA_pos_f[0, :], dt, peaks_RFA)
    vel_LFA = calc_velocity(LFA_pos_f[0, :], dt, peaks_LFA)
    vel_asym = asymmetry(vel_RFA, vel_LFA)
    vel_var = variability(safe_concatenate(vel_RFA, vel_LFA))

    # log dimensionless jerk — movement smoothness metric
    # more positive = more erratic/jerky movement
    ldj_RFA = calc_log_dimensionless_jerk(acc_RFA_f, RFA_pos_f[0, :], peaks_RFA, dt)
    ldj_LFA = calc_log_dimensionless_jerk(acc_LFA_f, LFA_pos_f[0, :], peaks_LFA, dt)
    ldj_asym = asymmetry(ldj_RFA, ldj_LFA)
    ldj_var = variability(safe_concatenate(ldj_RFA, ldj_LFA))

    # assemble results (include all variability entries)
    results = {
        "RFA_peaks": peaks_RFA,
        "LFA_peaks": peaks_LFA,
        "RFA_cycles": RFA_cycles,
        "LFA_cycles": LFA_cycles,
        "avg_RFA_cycle": avg_RFA_cycle,
        "avg_LFA_cycle": avg_LFA_cycle,
        "cycle_asymmetry": cycle_asym,
        "cycle_variability": cycle_variability,

        "swing_freq_RFA": swing_freq_RFA,
        "swing_freq_LFA": swing_freq_LFA,
        "swing_freq_asymmetry": swing_freq_asym,
        "swing_freq_variability": swing_freq_var,

        "lat_exc_RFA": lat_exc_RFA,
        "lat_exc_LFA": lat_exc_LFA,
        "lat_exc_asymmetry": lat_exc_asym,
        "lat_exc_variability": lat_exc_var,

        "sag_exc_RFA": sag_exc_RFA,
        "sag_exc_LFA": sag_exc_LFA,
        "sag_exc_asymmetry": sag_exc_asym,
        "sag_exc_variability": sag_exc_var,

        "vert_exc_RFA": vert_exc_RFA,
        "vert_exc_LFA": vert_exc_LFA,
        "vert_exc_asymmetry": vert_exc_asym,
        "vert_exc_variability": vert_exc_var,

        "peak_acc_RFA": peak_acc_RFA,
        "peak_acc_LFA": peak_acc_LFA,
        "peak_acc_asymmetry": peak_acc_asym,
        "peak_acc_variability": peak_acc_var,

        "rms_acc_RFA": rms_acc_RFA,
        "rms_acc_LFA": rms_acc_LFA,
        "rms_acc_asymmetry": rms_acc_asym,
        "rms_acc_variability": rms_acc_var,

        "vel_RFA": vel_RFA,
        "vel_LFA": vel_LFA,
        "vel_asymmetry": vel_asym,
        "vel_variability": vel_var,

        "ldj_RFA": ldj_RFA,
        "ldj_LFA": ldj_LFA,
        "ldj_asymmetry": ldj_asym,
        "ldj_variability": ldj_var,
    }
    return results

In [51]:
# pipeline to process all files

def batch_arm_analysis(folder_path, ext=".txt", plot=False, top_k=top_k, cutoff=None, per_frame_rel=False):
    files = get_files_from_folder(folder_path, ext)
    all_results = []

    for file_path in files:
        try:
            filename = os.path.basename(file_path)
            subject, trial_num, test_type = parse_filename(filename)

            result = arm_analysis(file_path, plot=plot, per_frame_rel=per_frame_rel, cutoff=cutoff)

            cycle_asym_nondir, cycle_asym_dir = unpack_asym(result.get("cycle_asymmetry"))
            swing_asym_nondir, swing_asym_dir = unpack_asym(result.get("swing_freq_asymmetry"))
            lat_exc_asym_nondir, lat_exc_asym_dir = unpack_asym(result.get("lat_exc_asymmetry"))
            sag_exc_asym_nondir, sag_exc_asym_dir = unpack_asym(result.get("sag_exc_asymmetry"))
            vert_exc_asym_nondir, vert_exc_asym_dir = unpack_asym(result.get("vert_exc_asymmetry"))
            vel_asym_nondir, vel_asym_dir = unpack_asym(result.get("vel_asymmetry"))
            rms_acc_asym_nondir, rms_acc_asym_dir = unpack_asym(result.get("rms_acc_asymmetry"))
            peak_acc_asym_nondir, peak_acc_asym_dir = unpack_asym(result.get("peak_acc_asymmetry"))
            ldj_asym_nondir, ldj_asym_dir = unpack_asym(result.get("ldj_asymmetry"))

            # build row
            row = {
                # IDs
                "subject": subject,
                "trial_num": trial_num,
                "test_type": test_type,

                # cycles & peaks
                #"avg_RFA_cycle": result.get("avg_RFA_cycle", np.nan),
                #"avg_LFA_cycle": result.get("avg_LFA_cycle", np.nan),
                #"cycle_asym_nondirectional": cycle_asym_nondir,
                #"cycle_asym_directional": cycle_asym_dir,
                #"cycle_variability": result.get("cycle_variability", np.nan),

                # swing frequency
                "swing_freq_RFA": result.get("swing_freq_RFA", np.nan),
                "swing_freq_LFA": result.get("swing_freq_LFA", np.nan),
                "swing_freq_asym_nondirectional": swing_asym_nondir,
                "swing_freq_asym_directional": swing_asym_dir,
                "swing_freq_variability": result.get("swing_freq_variability", np.nan),

                # lateral excursion
                "lat_exc_RFA_mean": mean_or_nan(result.get("lat_exc_RFA", [])),
                "lat_exc_RFA_variability": variability(result.get("lat_exc_RFA", [])),
                "lat_exc_LFA_mean": mean_or_nan(result.get("lat_exc_LFA", [])),
                "lat_exc_LFA_variability": variability(result.get("lat_exc_LFA", [])),
                "lat_exc_asym_nondirectional": lat_exc_asym_nondir,
                "lat_exc_asym_directional": lat_exc_asym_dir,
                "lat_exc_variability": result.get("lat_exc_variability", np.nan),

                # sagittal excursion
                "sag_exc_RFA_mean": mean_or_nan(result.get("sag_exc_RFA", [])),
                "sag_exc_RFA_variability": variability(result.get("sag_exc_RFA", [])),
                "sag_exc_LFA_mean": mean_or_nan(result.get("sag_exc_LFA", [])),
                "sag_exc_LFA_variability": variability(result.get("sag_exc_LFA", [])),
                "sag_exc_asym_nondirectional": sag_exc_asym_nondir,
                "sag_exc_asym_directional": sag_exc_asym_dir,
                "sag_exc_variability": result.get("sag_exc_variability", np.nan),

                # vertical excursion
                "vert_exc_RFA_mean": mean_or_nan(result.get("vert_exc_RFA", [])),
                "vert_exc_RFA_variability": variability(result.get("vert_exc_RFA", [])),
                "vert_exc_LFA_mean": mean_or_nan(result.get("vert_exc_LFA", [])),
                "vert_exc_LFA_variability": variability(result.get("vert_exc_LFA", [])),
                "vert_exc_asym_nondirectional": vert_exc_asym_nondir,
                "vert_exc_asym_directional": vert_exc_asym_dir,
                "vert_exc_variability": result.get("vert_exc_variability", np.nan),

                # velocity
                "vel_RFA_mean": mean_or_nan(result.get("vel_RFA", [])),
                "vel_RFA_variability": variability(result.get("vel_RFA", [])),
                "vel_LFA_mean": mean_or_nan(result.get("vel_LFA", [])),
                "vel_LFA_variability": variability(result.get("vel_LFA", [])),
                "vel_asym_nondirectional": vel_asym_nondir,
                "vel_asym_directional": vel_asym_dir,
                "vel_variability": result.get("vel_variability", np.nan),

                # root mean squared acceleration
                "rms_RFA_mean": mean_or_nan(result.get("rms_acc_RFA", [])),
                "rms_RFA_variability": variability(result.get("rms_acc_RFA", [])),
                "rms_LFA_mean": mean_or_nan(result.get("rms_acc_LFA", [])),
                "rms_LFA_variability": variability(result.get("rms_acc_LFA", [])),
                "rms_acc_asym_nondirectional": rms_acc_asym_nondir,
                "rms_acc_asym_directional": rms_acc_asym_dir,
                "rms_acc_variability": result.get("rms_acc_variability", np.nan),

                # peak acceleration
                "peak_acc_RFA_mean": mean_or_nan(result.get("peak_acc_RFA", [])),
                "peak_acc_RFA_variability": variability(result.get("peak_acc_RFA", [])),
                "peak_acc_LFA_mean": mean_or_nan(result.get("peak_acc_LFA", [])),
                "peak_acc_LFA_variability": variability(result.get("peak_acc_LFA", [])),
                "peak_acc_asym_nondirectional": peak_acc_asym_nondir,
                "peak_acc_asym_directional": peak_acc_asym_dir,
                "peak_acc_variability": result.get("peak_acc_variability", np.nan),
                
                # log mean squared jerk — movement smoothness
                "ldj_RFA_mean": mean_or_nan(result.get("ldj_RFA", [])),
                "ldj_RFA_variability": variability(result.get("ldj_RFA", [])),
                "ldj_LFA_mean": mean_or_nan(result.get("ldj_LFA", [])),
                "ldj_LFA_variability": variability(result.get("ldj_LFA", [])),
                "ldj_asym_nondirectional": ldj_asym_nondir,
                "ldj_asym_directional": ldj_asym_dir,
                "ldj_variability": result.get("ldj_variability", np.nan),
            }

            # add top-k peaks for key metrics
            selected_metrics = {
                "lat_exc": ("lat_exc_RFA", "lat_exc_LFA"),
                "sag_exc": ("sag_exc_RFA", "sag_exc_LFA"),
                "vert_exc": ("vert_exc_RFA", "vert_exc_LFA"),
                "vel": ("vel_RFA", "vel_LFA"),
                "peak_acc": ("peak_acc_RFA", "peak_acc_LFA"),
                "rms_acc": ("rms_acc_RFA", "rms_acc_LFA"),
                "ldj": ("ldj_RFA", "ldj_LFA"),
            }

            for mname, (rk, lk) in selected_metrics.items():
                arr_r = result.get(rk)
                arr_l = result.get(lk)
                peak_r = top_k_mean(arr_r, top_k)
                peak_l = top_k_mean(arr_l, top_k)
                row[f"{mname}_RFA_PEAK"] = peak_r
                row[f"{mname}_LFA_PEAK"] = peak_l
                peak_asym = asymmetry([peak_r], [peak_l])
                row[f"{mname}_PEAK_asym_nondirectional"] = peak_asym["nondirectional"]
                row[f"{mname}_PEAK_asym_directional"] = peak_asym["directional"]

            # these are renamed to "WORST3" and not "PEAK" because "worst" in this case does mean the highest jerk values
            row["ldj_RFA_WORST3_mean"] = row.pop("ldj_RFA_PEAK")
            row["ldj_LFA_WORST3_mean"] = row.pop("ldj_LFA_PEAK")
            row["ldj_WORST3_asym_nondirectional"] = row.pop("ldj_PEAK_asym_nondirectional")
            row["ldj_WORST3_asym_directional"] = row.pop("ldj_PEAK_asym_directional")

            all_results.append(row)

        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    df_results = pd.DataFrame(all_results)
    return df_results


## Result Export

These functions run the entire pipeline and save results into summary CSVs: one for ataxia, one for Parkinson's.

In [52]:
def save_results(df_results, output_file="arm_summary.csv"):
    df_results.to_csv(output_file, index=False)
    print(f"Results saved to {output_file}")

In [53]:
base_data_dir = Path("/Users/suraj/Desktop/daad/all_files_xsens")
output_dir = Path("/Users/suraj/Desktop/daad/code - june 2026/arm_results")

cohorts = ["ataxia", "pd", "healthy"]

for cohort in cohorts:
    folder_path = base_data_dir / f"all_files_xsens_{cohort}_included_in_study"
    results_df = batch_arm_analysis(folder_path, ext=".txt", plot=False, top_k=top_k, cutoff=filter_cutoff_frame_rate, per_frame_rel=True)
    output_csv = output_dir / f"{cohort}_arm_analysis_summary.csv"
    save_results(results_df, output_csv)

Results saved to /Users/suraj/Desktop/daad/code - june 2026/arm_results/ataxia_arm_analysis_summary.csv
Results saved to /Users/suraj/Desktop/daad/code - june 2026/arm_results/pd_arm_analysis_summary.csv
Results saved to /Users/suraj/Desktop/daad/code - june 2026/arm_results/healthy_arm_analysis_summary.csv


In [54]:
# Function to average RFA/LFA columns
def average_laterality(input_csv, output_csv):
    df = pd.read_csv(input_csv)

    # group numeric columns
    group_cols = ["subject", "trial_num", "test_type"]
    df_grouped = df.groupby(group_cols, as_index=False).mean(numeric_only=True)

    # detect right and left forearm pairs
    pair_map = {}

    for col in df_grouped.columns:
        if "RFA" in col:
            lfa_col = col.replace("RFA", "LFA")

            if lfa_col in df_grouped.columns:
                avg_col = col.replace("RFA", "avg")
                pair_map[avg_col] = (col, lfa_col)

    # create averaged columns
    for avg_col, (rfa_col, lfa_col) in pair_map.items():
        df_grouped[avg_col] = df_grouped[[rfa_col, lfa_col]].mean(axis=1)

    # drop original laterality columns
    cols_to_drop = [c for pair in pair_map.values() for c in pair]
    df_final = df_grouped.drop(columns=cols_to_drop)

    # save files
    save_results(df_final, output_csv)


# Process all datasets
base_dir = Path("/Users/suraj/Desktop/daad/code - june 2026/arm_results")

datasets = ["ataxia", "pd", "healthy"]

for dataset in datasets:
    input_csv = base_dir / f"{dataset}_arm_analysis_summary.csv"
    output_csv = base_dir / f"{dataset}_avg_arm_analysis_summary.csv"
    average_laterality(input_csv, output_csv)

Results saved to /Users/suraj/Desktop/daad/code - june 2026/arm_results/ataxia_avg_arm_analysis_summary.csv
Results saved to /Users/suraj/Desktop/daad/code - june 2026/arm_results/pd_avg_arm_analysis_summary.csv
Results saved to /Users/suraj/Desktop/daad/code - june 2026/arm_results/healthy_avg_arm_analysis_summary.csv
